# R3-M2: model adapters and LR ablation
Owner: Muneeb; reviewer: Subhan. This notebook validates the selected hidden128 epoch-16 BiGRU and the existing full LR as development bundles. It verifies BiGRU selection parity on all chunks and LR adapter parity on a 50-chunk sample. Calibration and operating thresholds remain R3-N1 work.

The approved two-fit LR ablation has already been completed. Existing evidence is checked and displayed; this notebook does not automatically repeat it or change an official model or lock.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import sys, os, subprocess, json
ROOT = Path('/content/drive/MyDrive/PAI&AIES Project')
SHARDS = ROOT / 'data/processed/cohort_v1'
FEATURES = ROOT / 'data/features/r2_s1_B'
OUTPUT = ROOT / 'runs/r3/r3_m2_adapters_verified_v3'
ABLATION = ROOT / 'experiments/r3_m2/runs/ablation01'
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT/'src')
os.environ['PYTHONDONTWRITEBYTECODE'] = '1'
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import torch
assert torch.cuda.is_available(), 'Use a T4 GPU for selected-checkpoint parity'
# Preserve the batch01 runtime for direct comparisons; do not silently upgrade.
from importlib.metadata import version
import platform
prior = json.loads((ROOT/'experiments/r3_m1/configs/batch01_resolved.json').read_text())
actual = {'python': platform.python_version(), 'torch': torch.__version__, 'numpy': version('numpy'), 'gpu': torch.cuda.get_device_name(0), 'cuda': torch.version.cuda}
assert actual == prior['environment'], (actual, prior['environment'])
assert all(version(k) == v for k,v in prior['dependencies'].items()), 'Dependency versions differ from batch01'
def run_command(args):
    tail=[]
    with subprocess.Popen([sys.executable, '-u', *map(str,args)], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
            tail=(tail+[line.rstrip()])[-30:]
        code=process.wait()
    if code:
        raise RuntimeError('Command failed:\n'+'\n'.join(tail))


## Fixture tests
These tests use disposable bundles and synthetic training fixtures. They require the selected batch01 environment and run the BiGRU checks on CPU and available CUDA. The temporary test report does not replace the original saved test evidence.


In [ ]:
import tempfile
import xml.etree.ElementTree as ET
with tempfile.TemporaryDirectory(prefix='r3_m2_tests_') as temp:
    junit = Path(temp) / 'fixtures.xml'
    run_command(['-m','pytest','-q','-p','no:cacheprovider',
                 'tests/test_r3_m2_adapters.py','tests/test_r3_m2_bigru_adapter.py',
                 'tests/test_r3_m2_ablation.py','tests/test_r3_m2_orchestration.py',
                 'tests/test_r2_bundle.py','--junitxml',junit])
    suites = ET.parse(junit).getroot().findall('.//testsuite')
    assert suites and all(int(s.get(k, 0)) == 0 for s in suites for k in ['failures', 'errors', 'skipped'])


## Development bundle parity
The existing verified output is checked against its evidence and artifact hashes. If the directory is absent, the packaging script can create it without fitting. If it exists without successful evidence, inspect that attempt; do not overwrite it. The selected checkpoint is hidden128, epoch 16.


In [ ]:
import hashlib

def check_evidence(directory, expected_status):
    evidence_file = directory / 'evidence.json'
    if not evidence_file.is_file():
        raise RuntimeError(f'Missing evidence in {directory}; inspect the existing attempt.')
    evidence = json.loads(evidence_file.read_text(encoding='utf-8'))
    if evidence.get('status') != expected_status or not evidence.get('artifacts'):
        raise RuntimeError(f'Incomplete or unexpected evidence in {directory}.')
    for name, expected in evidence['artifacts'].items():
        path = (directory / name).resolve()
        if not path.is_relative_to(directory.resolve()) or not path.is_file():
            raise RuntimeError(f'Missing or unsafe artifact: {name}')
        if hashlib.sha256(path.read_bytes()).hexdigest() != expected:
            raise RuntimeError(f'Artifact checksum mismatch: {name}')
    return evidence

if not OUTPUT.exists():
    run_command([ROOT/'scripts/r3_m2_package.py','--output',OUTPUT,'--shard-dir',SHARDS,'--device','cuda'])
packaging = check_evidence(OUTPUT, 'development_bundle_parity_passed')
assert packaging['selected_checkpoint_sha256'] == '435d47b36431e7f0f6b6f1bdd2459c8a6d06074b79fcc7ff69631c0c93cc687e'
print(json.dumps(packaging, indent=2))


## Review completed LR ablation
The approved full-67 versus reduced-60 feature comparison has already consumed its two-fit allowance. The cell below verifies and displays the existing evidence. If it is missing or incomplete, inspect the previous attempt and authorization; do not restart or create a replacement run automatically.


In [ ]:
if not ABLATION.exists():
    print('Ablation output is absent. Do not rerun an already-consumed fit budget; locate the existing results and review authorization.')
else:
    ablation = check_evidence(ABLATION, 'experimental_ablation_complete')
    print(json.dumps(ablation, indent=2))
    print('\nComparison:\n')
    print((ABLATION/'comparison.md').read_text(encoding='utf-8'))


The verified bundle evidence is in `runs/r3/r3_m2_adapters_verified_v3/`. The ablation evidence and coefficients are in `experiments/r3_m2/runs/ablation01/`, with a separate `experiments/r3_m2/ledger.csv`. These development artifacts require Subhan's review and do not replace R3-N1 calibration or an official lock.
